# IP/TCP traffic simulation and generation

# Step 1 - Upload a CSV File

In [ ]:
import csv

csv_file = 'data_input_main.csv'

messages = []
with open(csv_file, newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        messages.append(row)

print('Number of messages:', len(messages))
print('Columns:', list(messages[0].keys()))
print()
for row in messages[:5]:
    print(row)

## Step 2 - Simulating data packaging in IP/TCP layers

In [ ]:
CLIENT_IP = '127.0.0.1'
SERVER_IP = '127.0.0.1'
CLIENT_PORT = 51515
SERVER_PORT = 8080

encapsulated_packets = []

for row in messages:
    source_is_client = row['src_app'] == 'client_browser'

    packet = {
        'application_layer': {
            'protocol': row['app_protocol'],
            'message': row['message']
        },
        'tcp_layer': {
            'src_port': CLIENT_PORT if source_is_client else SERVER_PORT,
            'dst_port': SERVER_PORT if source_is_client else CLIENT_PORT,
            'transport_protocol': 'TCP'
        },
        'ip_layer': {
            'src_ip': CLIENT_IP if source_is_client else SERVER_IP,
            'dst_ip': SERVER_IP if source_is_client else CLIENT_IP,
            'network_protocol': 'IP'
        },
        'timestamp': row['timestamp']
    }

    encapsulated_packets.append(packet)

for i, packet in enumerate(encapsulated_packets[:5], start=1):
    print(f'Packet {i}')
    print('  Application data:', packet['application_layer'])
    print('  TCP layer:', packet['tcp_layer'])
    print('  IP layer:', packet['ip_layer'])
    print('  Timestamp:', packet['timestamp'])
    print()


## Step 3 - HTTP traffic capture and traffic generation (WIRESHARK)

In [ ]:
import http.server
import socketserver
import threading
import socket
import time

HTTP_HOST = '127.0.0.1'
HTTP_PORT = 8080

class DemoHttpHandler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == '/about.html':
            self.send_response(404)
            self.end_headers()
            self.wfile.write(b'404 Not Found')
        else:
            self.send_response(200)
            self.end_headers()
            self.wfile.write(b'Hello from demo HTTP server')

    def log_message(self, format, *args):
        # Prevent long server logs inside the notebook
        pass

try:
    httpd = socketserver.TCPServer((HTTP_HOST, HTTP_PORT), DemoHttpHandler)
    server_thread = threading.Thread(target=httpd.serve_forever, daemon=True)
    server_thread.start()
    print(f'Demo HTTP server started on {HTTP_HOST}:{HTTP_PORT}')
except OSError:
    print('HTTP server may already be running on port 8080')

time.sleep(1)

for row in messages:
    if row['message'].startswith('GET'):
        path = row['message'].split(' ')[1]
        request = f'GET {path} HTTP/1.1\r\nHost: localhost\r\nConnection: close\r\n\r\n'

        with socket.create_connection((HTTP_HOST, HTTP_PORT)) as sock:
            sock.sendall(request.encode())
            response = sock.recv(1024).decode(errors='ignore')
            status_line = response.splitlines()[0] if response else 'No response'
            print(row['message'], '->', status_line)

print('Traffic was generated. Stop Wireshark capture and save the pcap file.')